## Jupyter Notebook M3 for modeling
*Hypothesis*: I can predict the country from a google maps street view image.

This code block uses an S3 bucket I set up using the AWS CLI. It creates a list of shards; 5 test shards
which are formatted as "images/test/{i:02d}.zip", and 10 training shards formatted as "images/train/{i:02d}.zip".

For each shard in the list, we first check if it already exists in the S3 bucket. If it does not, we dowload
the shard locally and upload it to the S3 bucket using huggingface. It is then removed from local storage.

In [ ]:
import os, boto3
from huggingface_hub import hf_hub_download

BUCKET = "worldguessr-ml-data"
REPO = "osv5m/osv5m"
NUM_TRAIN_SHARDS = 10

shards = [f"images/test/{i:02d}.zip" for i in range(5)]
shards += [f"images/train/{i:02d}.zip" for i in range(NUM_TRAIN_SHARDS)]

s3 = boto3.client("s3")
for f in shards:
    key = f"raw/{f}"
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        print("already in S3, skipping:", key, flush=True)
        continue
    except s3.exceptions.ClientError:
        pass
    path = hf_hub_download(REPO, filename=f, repo_type="dataset", local_dir="data/raw")
    print("uploading", key, flush=True)
    s3.upload_file(path, BUCKET, key)
    os.remove(path)
    print("done", key, flush=True)
print("ALL DONE", flush=True)

This next code block converts the zip shards in S3 into 224x224 records for effeciency of training the model.
Each record stores JPEG bytes, country code, and image id. These files are stored both locally and in s3. 

In [ ]:
import io, os, json, zipfile #reads shards without extracting them to disk
from collections import Counter #tally images per country
from multiprocessing import Pool #runs multiple shards at once
import boto3, pandas as pd #talk to s3, read csvs
from PIL import Image #decodes and resizes images

BUCKET = "worldguessr-ml-data"
SIZE = 224 #native output resolution of CLIP
OUT_PREFIX = f"processed/tfrecords-{SIZE}" 
LOCAL_OUT = f"data/processed/tfrecords-{SIZE}"
WORK = "data/work" #scratch folder for processing
SHARDS = [("test", i) for i in range(5)] + [("train", i) for i in range(10)] #list of training and test shards.

# Loaded once in the parent; forked workers share it
META = {}
for split in ("train", "test"):
    df = pd.read_csv(f"data/raw/{split}.csv", usecols=["id", "country"], dtype={"id": str})
    META[split] = dict(zip(df.id, df.country))
    print(f"loaded {split} metadata: {len(df)} rows", flush=True)

def process_shard(job):
    import tensorflow as tf  # imported inside the worker
    split, idx = job
    s3 = boto3.client("s3")
    name = f"shard-{idx:02d}"
    rec_key = f"{OUT_PREFIX}/{split}/{name}.tfrecord"
    cnt_key = f"{OUT_PREFIX}/{split}/{name}.counts.json"

    try:  # already done on a previous run
        s3.head_object(Bucket=BUCKET, Key=rec_key)
        body = s3.get_object(Bucket=BUCKET, Key=cnt_key)["Body"].read()
        print(f"skip {split}/{name} (already in S3)", flush=True)
        return split, json.loads(body)
    except s3.exceptions.ClientError:
        pass

    os.makedirs(WORK, exist_ok=True)
    os.makedirs(f"{LOCAL_OUT}/{split}", exist_ok=True)
    zip_path = f"{WORK}/{split}-{idx:02d}.zip"
    rec_path = f"{LOCAL_OUT}/{split}/{name}.tfrecord"
    s3.download_file(BUCKET, f"raw/images/{split}/{idx:02d}.zip", zip_path)

    def b(v): return tf.train.Feature(bytes_list=tf.train.BytesList(value=[v]))
    counts, bad, nolabel = Counter(), 0, 0
    with zipfile.ZipFile(zip_path) as z, tf.io.TFRecordWriter(rec_path) as w:
        for member in z.namelist():
            if not member.lower().endswith(".jpg"):
                continue
            img_id = os.path.basename(member)[:-4]
            country = META[split].get(img_id)
            if country is None:
                nolabel += 1
                continue
            try:
                # Squash to a square (keeps the whole scene, slight aspect distortion)
                img = Image.open(io.BytesIO(z.read(member))).convert("RGB")
                img = img.resize((SIZE, SIZE), Image.BICUBIC)
                buf = io.BytesIO()
                img.save(buf, format="JPEG", quality=90)
            except Exception:
                bad += 1
                continue
            ex = tf.train.Example(features=tf.train.Features(feature={
                "image": b(buf.getvalue()),
                "country": b(country.encode()),
                "id": b(img_id.encode()),
            }))
            w.write(ex.SerializeToString())
            counts[country] += 1

    s3.upload_file(rec_path, BUCKET, rec_key)
    s3.put_object(Bucket=BUCKET, Key=cnt_key, Body=json.dumps(counts).encode())
    os.remove(zip_path)
    print(f"done {split}/{name}: {sum(counts.values())} images, "
          f"{bad} unreadable, {nolabel} without label", flush=True)
    return split, dict(counts)

if __name__ == "__main__":
    totals = {"train": Counter(), "test": Counter()}
    with Pool(4) as pool:
        for split, counts in pool.imap_unordered(process_shard, SHARDS):
            totals[split].update(counts)

    os.makedirs("data/processed", exist_ok=True)
    for split, c in totals.items():
        with open(f"data/processed/counts_{split}.json", "w") as f:
            json.dump(dict(c.most_common()), f, indent=1)
        print(f"{split}: {sum(c.values())} images, {len(c)} countries", flush=True)

    tr = totals["train"]
    for n in [50, 100, 200, 500]:
        kept = [k for k, v in tr.items() if v >= n]
        test_kept = sum(totals["test"][k] for k in kept)
        print(f"threshold {n}: {len(kept)} countries, "
              f"{test_kept}/{sum(totals['test'].values())} test images kept", flush=True)
    print("ALL DONE", flush=True)


This code block trains the multinomial logistic regression model with a single dense layer and a softmax activation function
on 64x64 raw pixels. 

In [ ]:
"""Multinomial logistic regression (single Dense + softmax) on 64x64 raw pixels."""
import json, os, glob
import boto3, tensorflow as tf

BUCKET = "worldguessr-ml-data"
MIN_COUNT = 100       # keep countries with >= this many training images
IMG = 64              # downsample 224 -> 64
BATCH = 512
EPOCHS = 15
LR = 1e-3
EVAL_TEST = True      # set False while tuning

DATA = "data/processed"
REC = f"{DATA}/tfrecords-224"
RUN = f"data/runs/logreg_{IMG}px_min{MIN_COUNT}"
os.makedirs(RUN, exist_ok=True)

#Label mapping, shared with the LoRA model later
counts_train = json.load(open(f"{DATA}/counts_train.json"))
counts_test = json.load(open(f"{DATA}/counts_test.json"))
countries = sorted(c for c, n in counts_train.items() if n >= MIN_COUNT)
c2i = {c: i for i, c in enumerate(countries)}
map_path = f"{DATA}/country_to_idx_min{MIN_COUNT}.json"
json.dump(c2i, open(map_path, "w"), indent=1)
boto3.client("s3").upload_file(map_path, BUCKET, f"processed/country_to_idx_min{MIN_COUNT}.json")
NUM_CLASSES = len(countries)
print(f"{NUM_CLASSES} classes", flush=True)

table = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        tf.constant(countries), tf.range(NUM_CLASSES, dtype=tf.int64)),
    default_value=-1)

FEATURES = {"image": tf.io.FixedLenFeature([], tf.string),
            "country": tf.io.FixedLenFeature([], tf.string)}

def parse(record):
    f = tf.io.parse_single_example(record, FEATURES)
    label = table.lookup(f["country"])
    img = tf.io.decode_jpeg(f["image"], channels=3)
    img = tf.image.resize(img, (IMG, IMG), antialias=True)
    return tf.cast(tf.round(img), tf.uint8), label   # uint8 keeps the cache small

def make_ds(files, cache_name=None, shuffle=False):
    ds = tf.data.TFRecordDataset(files, num_parallel_reads=tf.data.AUTOTUNE)
    ds = ds.map(parse, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.filter(lambda x, y: y >= 0)                # drop countries below cutoff
    if cache_name:
        ds = ds.cache(f"{RUN}/cache_{cache_name}")
    if shuffle:
        ds = ds.shuffle(50_000)
    ds = ds.batch(BATCH).map(lambda x, y: (tf.cast(x, tf.float32), y))
    return ds.prefetch(tf.data.AUTOTUNE)

train_files = sorted(glob.glob(f"{REC}/train/shard-0[0-8].tfrecord"))
val_files = [f"{REC}/train/shard-09.tfrecord"]
test_files = sorted(glob.glob(f"{REC}/test/*.tfrecord"))

train_ds = make_ds(train_files, "train", shuffle=True)
val_ds = make_ds(val_files, "val")

# Model: normalize pixels, flatten, one Dense softmax layer
norm = tf.keras.layers.Normalization(axis=-1)
norm.adapt(make_ds(train_files[:1]).take(40).map(lambda x, y: x))  # ~20k images

model = tf.keras.Sequential([
    tf.keras.Input((IMG, IMG, 3)),
    norm,
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")],
)
model.summary()

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                         restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(f"{RUN}/history.csv"),
    ],
)
model.save(f"{RUN}/model.keras")

# Results
majority = max(countries, key=lambda c: counts_train[c])
test_kept = sum(counts_test.get(c, 0) for c in countries)
results = {"majority_class": majority,
           "majority_baseline_test_acc": counts_test.get(majority, 0) / test_kept}
if EVAL_TEST:
    loss, acc, top5 = model.evaluate(make_ds(test_files))
    results.update(test_loss=loss, test_acc=acc, test_top5=top5)
print(json.dumps(results, indent=1), flush=True)
json.dump(results, open(f"{RUN}/results.json", "w"), indent=1)

s3 = boto3.client("s3")
for name in ["model.keras", "history.csv", "results.json"]:
    s3.upload_file(f"{RUN}/{name}", BUCKET, f"runs/{os.path.basename(RUN)}/{name}")
print("ALL DONE", flush=True)

In [11]:
#Results of the logistic regression model
import pandas as pd 
import matplotlib.pyplot as plt

results = "../data/runs/logreg_64px_min100/history.csv"

plotting_data = pd.read_csv(results)

epochs =  plotting_data['epoch']
training_loss = plotting_data['loss']
val_loss = plotting_data['val_loss']

plt.figure(10,6)

plt.plot(epochs, training_loss, label="Training Loss", marker="o")
plt.plot(epochs, val_loss, label="Validation Loss", marker="o")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Logistic Regression: Training vs. Validation Loss")

plt.legend()
plt.grid(alpha=0.3)
plt.xticks(epochs)
plt.tight_layout()
plt.show()

TypeError: Value after * must be an iterable, not int